<a href="https://colab.research.google.com/github/Tuchobm/Curso-IA-Google-Colab/blob/main/5_1_inferencia_result.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 1: Inferencia de LLM

En este ejercicio, aprenderás a cargar y utilizar un modelo de lenguaje pre-entrenado de la biblioteca 'transformers' para realizar inferencia (generación de texto).

Configuraremos un pipeline de generación de texto e interactuaremos con el modelo a través de un prompt de sistema y entradas de usuario.

Además, encontrarás partes del código contendrán el comentario de '# ACTIVIDAD' que indican dónde debes completar el código o realizar tareas específicas. Asegúrate de seguir las instrucciones y completar el código donde se indique. Finalmente, en el final de este ejercicio, deberás responder a una serie de preguntas.

# Cargar el modelo y el tokenizer

In [ ]:
from transformers import pipeline

# ACTIVIDAD: Prueba diferentes modelos de lenguaje para ver cuál se adapta mejor a tus necesidades.
model_name = "unsloth/Llama-3.2-1B-Instruct"
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
# model_name = "Qwen/Qwen2.5-7B-Instruct-1M"

model = pipeline(
    task="text-generation",
    model=model_name,
    torch_dtype="auto",
    device_map="auto",
)

Some parameters are on the meta device because they were offloaded to the disk and cpu.
Device set to use cpu


## Conversación con el modelo

In [ ]:
# ACTIVIDAD: Prueba diferentes system prompts y hyperparámetros
system_prompt = """
Eres un assistente virtual que ayuda a los usuarios a encontrar información sobre la historia de España.
Pero solo conces información sobre la historia de España antes del 1800.
Intenta siempre responder com una persona de esa época.
"""
temperature = 0.8
top_p = 0.95
max_new_tokens =

messages = [{"role": "system", "content": system_prompt}]
while True:
    user_input = input("Usuario (Escribe 'exit' para salir): ")
    if user_input.lower() == "exit" or user_input.lower() == "":
        break

    messages.append({"role": "user", "content": user_input})
    response = model(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=model.tokenizer.eos_token_id,
    )
    msg = response[0]["generated_text"][-1]

    # Encuentra el pensamiento del asistente (deepseek)
    if "</think>" in msg["content"]:
        pensamiento, msg["content"] = msg["content"].split("</think>", 1)
        pensamiento, msg["content"] = pensamiento.strip(), msg["content"].strip()
        print("Pensamiento asistente:", pensamiento)
    messages.append(msg)
    print("Asistente:", msg["content"])

Asistente: Bienvenido a mi estudio de historia de España. Me alegra que hayas decidido explorar la rica y fascinante tradición de nuestro país. ¿Cuál es tu interés específico sobre la historia de España antes del año 1800? ¿Buscas saber sobre los motivos por los que se declaró la independencia de España, la batalla de San Julián de los Caballeros o quizás sobre la vida en el reino de los reyes Católicos?

La historia de España antes del siglo XVIII es un fascinante tema que ofrece numerosos capítulos interesantes y complejos. ¿Tienes algún tema específico en mente?


## Preguntas

- Describe cómo cambiar el `system_prompt` influye en la personalidad, el tono y la información que proporciona el asistente. Busca 3 ejemplos para que el modelo actúe como un asistente diferente (p. ej., un amigo, un experto en historia, un profesor). ¿Cómo cambia la calidad de las respuestas? ¿Qué tipo de preguntas funcionan mejor con cada `system_prompt`?
  - `Eres mi amigo y me hablas de forma cercana y coloquial. Usa anécdotas y ejemplos cotidianos.`
    - Tipo de respuesta: tono informal, emoticonos, comparaciones de la vida diaria.
    - Preguntas ideales: Preguntas sobre experiencias personales, consejos de vida, temas cotidianos.
  - `Eres un historiador escéptico que cuestiona las fuentes oficiales. Señala incertidumbres y posibles sesgos.`
    - Tipo de respuesta: análisis crítico, referencias a fuentes históricas, cuestionamiento de narrativas populares.
    - Preguntas ideales: Preguntas sobre eventos históricos, análisis de fuentes, teorías de conspiración.
  - `Eres un profesor universitario de inteligencia Artifical especializado en el Aprendizaje Profundo. Explica con claridad y ofrece ejemplos, cita fuentes cuando sea posible.`
    - Tipo de respuesta: explicaciones detalladas, referencias a investigaciones, ejemplos prácticos.
    - Preguntas ideales: Preguntas sobre conceptos técnicos, teorías de aprendizaje profundo, aplicaciones prácticas.

- Experimenta con diferentes valores para `temperature` (p. ej., 0.2, 0.7, 1.0) y `top_p` (p. ej., 0.5, 0.95). ¿Cómo afectan estos parámetros a la creatividad frente a la coherencia del texto generado? ¿Qué sucede si reduces significativamente `max_new_tokens`?

  - Depende mucho del modelo, pero en general:
  - **Temperature**  
    - 0.2: muy determinista, repite patrones aprendidos, coherencia alta, creatividad baja.  
    - 0.7: buen equilibrio entre fluidez y novedad.
    - 1.0: alta creatividad, posibles incoherencias o errores.

  - **Top_p**  
    - 0.5: sólo usa el 50 % de probabilidad acumulada, reduce vocabulario “inusual”, más conservador.  
    - 0.95: deja entrar más tokens poco probables, aumenta variedad.

  - **max_new_tokens**  
    - Valores bajos (p. ej. 20): respuestas muy concisas o truncadas.  
    - Valores altos (p. ej. 256): respuestas largas, pero riesgo de divagar o repetirse.

- ¿El modelo se adhiere estrictamente a las restricciones establecidas en el `system_prompt` (p. ej., salirse de su rol)? ¿Cómo responde a preguntas que no están directamente relacionadas con su rol? ¿Qué tipo de preguntas parecen funcionar mejor?
  - Generalmente, el modelo sigue las restricciones del `system_prompt`, pero puede desviarse completamente de su rol si la pregunta es muy directa o provocativa. Preguntas que intentan modificar el comportamiento del modelo ("prompt jailbreak") pueden llevar a que el modelo ignore instrucciones previas. Preguntas que funcionan mejor son aquellas que están alineadas con el rol definido en el `system_prompt` y que permiten al modelo ofrecer respuestas relevantes y útiles.
  
- Hazle al modelo la misma pregunta usando diferentes formulaciones (p. ej., una pregunta simple frente a una más detallada). ¿Cómo afecta la calidad o el detalle de la entrada del usuario a la respuesta del asistente?
  - Generalmente, preguntas más detalladas o específicas generan respuestas más completas y relevantes. Preguntas vagas pueden llevar a respuestas generales o irrelevantes. Pero no siempre es así. A veces, preguntas simples pueden generar respuestas más creativas o inesperadas.

- Continúa una conversación durante varios turnos. ¿Parece el modelo "recordar" partes anteriores de la conversación? ¿Cómo podría la ventana de contexto limitada afectar las interacciones más largas?
  - Mayoritariamente, el modelo recuerda partes de la conversación, pero puede perder detalles importantes si la conversación se alarga demasiado y se sale de la ventana de contexto.

- ¿Qué diferencias encuentras entre los tres modelos? ¿Cuál parece ser más efectivo para la generación de texto? ¿Por qué crees que es así?
  - Deepseek: Modelo de razonamiento, piensa antes de responder. Su pensamiento acaba después de `</think>`
  - Llama-3.2. y Qwen2: Relativamente similares y dependiendo de los hiperpparámetros, pueden ser más creativos o más coherentes.